# Setup

In [2]:
!pip install wandb --quiet

import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [ ]:
run = wandb.init(
    entity="23f2002974-dl-genai-project",
    project="23f2002974-t12026",
    name="Resnet_pretrained_CNN_Run2",
    config={
        "architecture": "Resnet CNN",
        "epochs": 25,
    }
)

# Libraries and imports

In [1]:
import os
import random

import numpy as np
import pandas as pd
from tqdm import tqdm

import librosa
import torchaudio.transforms as T

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.models as models


from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


# Data Pre-processing & Feature Extraction

In [ ]:
"""Config"""
# Standard audio settings
SR = 22050
N_MELS = 128
DURATION = 30 

# Where the data is coming from
GENRE_INPUT = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
NOISE_INPUT = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"

# Where we want to save the processed files
GENRE_OUTPUT = "/kaggle/working/processed_mels"
NOISE_OUTPUT = "/kaggle/working/processed_noise_mels"

STEM_FILES = ["drums.wav", "bass.wav", "vocals.wav", "other.wav"]

In [ ]:
def preprocess_dataset():
    # Create the main output directory
    os.makedirs(GENRE_OUTPUT, exist_ok=True)

    all_items = os.listdir(GENRE_INPUT)
    genres = []
    for item in all_items:
        if os.path.isdir(os.path.join(GENRE_INPUT, item)):
            genres.append(item)

    for genre in genres:
        genre_input_path = os.path.join(GENRE_INPUT, genre)
        genre_output_path = os.path.join(GENRE_OUTPUT, genre)
        os.makedirs(genre_output_path, exist_ok=True)

        '''Get list of songs in the genre folder'''
        all_songs = os.listdir(genre_input_path)
        songs = []
        for s in all_songs:
            if os.path.isdir(os.path.join(genre_input_path, s)):
                songs.append(s)

        for song in tqdm(songs, desc=f"Processing {genre}"):
            song_input = os.path.join(genre_input_path, song)
            song_output = os.path.join(genre_output_path, song)
            os.makedirs(song_output, exist_ok=True)

            for stem in STEM_FILES:
                input_file = os.path.join(song_input, stem)
                output_file = os.path.join(song_output, stem.replace(".wav", ".npy"))

                '''Skip if already processed'''
                if os.path.exists(output_file):
                    continue

                try:
                    '''Load audio with your Config settings (SR and DURATION)'''
                    
                    audio, _ = librosa.load(input_file, sr=SR, duration=DURATION)

                    '''Pad if audio is shorter than 30s'''
                    
                    target_len = SR * DURATION
                    if len(audio) < target_len:
                        audio = np.pad(audio, (0, target_len - len(audio)))

                    
                    '''Compute Mel Spectrogram using your N_MELS'''
                    
                    mel = librosa.feature.melspectrogram(
                        y=audio,
                        sr=SR,
                        n_mels=N_MELS,
                        n_fft=2048,
                        hop_length=512
                    )


                    np.save(output_file, mel.astype(np.float32))
                except Exception as e:
                    print(f"Error processing {input_file}: {e}")

In [ ]:
preprocess_dataset()

In [ ]:
def preprocess_noise():
    
    '''Create the output folder using your defined NOISE_OUTPUT'''
    os.makedirs(NOISE_OUTPUT, exist_ok=True)

    '''Get all wav files from NOISE_INPUT'''
    noise_files = []
    for f in os.listdir(NOISE_INPUT):
        if f.endswith(".wav"):
            noise_files.append(f)
    print(f"Processing {len(noise_files)} noise files")

    for file in tqdm(noise_files):
        input_path = os.path.join(NOISE_INPUT, file)
        output_path = os.path.join(NOISE_OUTPUT, file.replace(".wav", ".npy"))


        if os.path.exists(output_path):
            continue

        try:
            '''Load audio using your SR (Sample Rate)'''
            audio, _ = librosa.load(input_path, sr=SR)

            '''Convert to Mel Spectrogram using your N_MELS'''
            mel = librosa.feature.melspectrogram(
                y=audio,
                sr=SR,
                n_mels=N_MELS,
                n_fft=2048,
                hop_length=512
            )

            np.save(output_path, mel.astype(np.float32))

        except Exception as e:
            print(f"Failed to process {file}: {e}")

In [ ]:
preprocess_noise()

# Custom Dataset :

In [1]:
class MashupDataset(Dataset):

    def __init__(self, processed_dir, noise_dir, song_dict, genres, size=30000, is_val=False):
        
        self.processed_dir = processed_dir
        self.song_dict = song_dict
        self.genres = genres
        self.size = size
        self.is_val = is_val

        ''' Create a mapping from genre name to a number '''
        self.genre_to_idx = {}
        for i, g in enumerate(genres):
            self.genre_to_idx[g] = i

        ''' Get all noise files using a simple loop '''
        self.noise_files = []
        for f in os.listdir(noise_dir):
            if f.endswith(".npy"):
                full_path = os.path.join(noise_dir, f)
                self.noise_files.append(full_path)

        ''' Setup masking for SpecAugment '''
        self.freq_mask = T.FrequencyMasking(freq_mask_param=30)
        self.time_mask = T.TimeMasking(time_mask_param=80)
        
        ''' Set target size for 30 seconds of audio '''
        self.target_h = 128
        self.target_w = 1292
        self.crop_size = 1292

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        
        ''' Pick a genre. Sequential for validation, random for training '''
        if self.is_val:
            genre = self.genres[idx % len(self.genres)]
        else:
            genre = random.choice(self.genres)
            
        songs = self.song_dict[genre]
        mixed = np.zeros((self.target_h, self.target_w), dtype=np.float32)
        stems = ["drums.npy", "bass.npy", "vocals.npy", "other.npy"]

        ''' --- Stem Mixing --- '''
        for stem in stems:
            
            ''' During training, randomly skip some stems to make model stronger '''
            if not self.is_val:
                if random.random() < 0.1:
                    continue

            ''' Pick a random song for this specific stem '''
            random_song = random.choice(songs)
            path = os.path.join(self.processed_dir, genre, random_song, stem)
            mel = np.load(path)

            ''' Make sure the width matches our target '''
            if mel.shape[1] > self.target_w:
                mel = mel[:, :self.target_w]
            else:
                padding = self.target_w - mel.shape[1]
                mel = np.pad(mel, ((0, 0), (0, padding)))

            ''' Apply random volume gain only during training '''
            gain = 1.0
            if not self.is_val:
                gain = random.uniform(0.4, 1.6)
            
            mixed = mixed + (mel * gain)

        ''' --- Training-Only Augmentations --- '''
        if not self.is_val:
            
            ''' Time Shifting '''
            shift = random.randint(-200, 200)
            mixed = np.roll(mixed, shift, axis=1)

            ''' Adding Random Silence '''
            if random.random() < 0.3:
                sil_len = random.randint(100, 300)
                start = random.randint(0, self.target_w - sil_len)
                mixed[:, start : start + sil_len] = 0

            ''' Adding Background Noise '''
            num_noises = random.randint(2, 5)
            for _ in range(num_noises):
                noise_path = random.choice(self.noise_files)
                noise = np.load(noise_path)

                ''' Standardize noise width '''
                if noise.shape[1] >= self.target_w:
                    noise = noise[:, :self.target_w]
                    noise_start = 0
                else:
                    noise_start = random.randint(0, self.target_w - noise.shape[1])

                ''' Calculate noise power for SNR scaling '''
                snr = random.uniform(8, 25)
                sig_p = mixed.mean() + 1e-8
                noi_p = noise.mean() + 1e-8
                scale = sig_p / ( (10**(snr/10)) * noi_p )
                
                end_pos = noise_start + noise.shape[1]
                mixed[:, noise_start : end_pos] += (noise * scale)

        ''' Final Processing (Always Done)'''
        
        ''' Convert to Decibels and Normalize '''
        mel_db = librosa.power_to_db(np.maximum(mixed, 1e-10))
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
        
        ''' Convert to Torch Tensor '''
        mel_tensor = torch.tensor(mel_db).unsqueeze(0)

        ''' --- SpecAugment Only for Training --- '''
        if not self.is_val:
            mel_tensor = self.freq_mask(mel_tensor)
            mel_tensor = self.time_mask(mel_tensor)

        label = self.genre_to_idx[genre]
        return mel_tensor, label

NameError: name 'Dataset' is not defined

# Train / Test Split

In [ ]:
def prepare_stratified_split(root_dir, val_total=3000):
    ''' 1. Find and sort all genre folders in the processed directory '''
    all_genres = os.listdir(root_dir)
    genres = []
    for g in all_genres:
        if os.path.isdir(os.path.join(root_dir, g)):
            genres.append(g)
    genres.sort()

    ''' 2. Calculate how many songs we want for validation per genre '''
    num_genres = len(genres)
    val_target_per_genre = val_total // num_genres
    
    train_songs_dict = {}
    val_songs_dict = {}

    ''' 3. Loop through each genre to split the data '''
    for genre in genres:
        genre_path = os.path.join(root_dir, genre)
        
        ''' Get all song subfolders inside this genre '''
        all_contents = os.listdir(genre_path)
        songs = []
        for item in all_contents:
            if os.path.isdir(os.path.join(genre_path, item)):
                songs.append(item)
        songs.sort()
        
        total_available = len(songs)
        
        ''' 
        Safety Check: We only take up to 20% for validation. 
        This ensures we never run out of training data.
        '''
        max_val_allowed = int(total_available * 0.2)
        
        ''' Determine how many to actually put in validation '''
        count_for_val = val_target_per_genre
        if count_for_val > max_val_allowed:
            count_for_val = max_val_allowed
            
        ''' Ensure at least one song is for validation if possible '''
        if count_for_val == 0 and total_available > 0:
            count_for_val = 1
            
        ''' Split the list of songs '''
        val_songs_dict[genre] = songs[:count_for_val]
        train_songs_dict[genre] = songs[count_for_val:]
        
        ''' Print the results for clarity '''
        print(f"Genre {genre}: Total={total_available}, Train={len(train_songs_dict[genre])}, Val={len(val_songs_dict[genre])}")
        
        ''' Safety check to prevent errors later '''
        if len(train_songs_dict[genre]) == 0:
            print(f"WARNING: Genre {genre} has no training songs. Please reduce val_total.")

    return train_songs_dict, val_songs_dict, genres

In [ ]:
train_songs, val_songs, genres = prepare_stratified_split(PROCESSED_DIR, val_total=5000)

# Data Loaders

In [ ]:
train_dataset = MashupDataset(GENRE_OUTPUT, NOISE_OUTPUT, train_songs, genres, size=60000)
val_dataset = MashupDataset(GENRE_OUTPUT, NOISE_OUTPUT, val_songs, genres, size=6000, is_val=True)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

# Model Architecture

## Resnet18 pretrianed

In [ ]:



class ResNet18Audio(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        # load pretrained resnet
        self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # change input layer (RGB → 1 channel spectrogram)
        self.model.conv1 = nn.Conv2d(
            1,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        # replace classifier
        in_features = self.model.fc.in_features

        self.model.fc = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )


    def forward(self, x):

        return self.model(x)

In [ ]:
model = DeepAudioCNN(num_classes=10).to(device)

# Loss Function, Optimizer, and Metrics

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=20)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

scaler = torch.amp.GradScaler("cuda")


# Training Loop definition

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    pbar = tqdm(loader)

    for x, y in pbar:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        with autocast("cuda"):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

        pbar.set_description(
            f"Train Loss {total_loss/(total/loader.batch_size):.4f} Acc {100*correct/total:.2f}%"
        )

    avg_loss = total_loss / len(loader)
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1

# Validation Loop definition

In [ ]:
def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x, y in tqdm(loader):

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1

# Model training and WndB logging

In [ ]:
def train_model(model,train_loader,val_loader,criterion,optimizer,scheduler,scaler,device,epochs,patience):
    best_val_acc = 0
    patience_counter = 0

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")

        # -------- TRAIN --------
        train_loss, train_acc, train_f1 = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            device
        )

        # -------- VALIDATE --------
        val_loss, val_acc, val_f1 = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )

        print(
            f"\nEpoch [{epoch+1}/{epochs}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Train Acc: {train_acc:.4f} "
            f"| Train F1: {train_f1:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Val Acc: {val_acc:.4f} "
            f"| Val F1: {val_f1:.4f}"
        )

        # -------- SCHEDULER --------
        scheduler.step(val_acc)

        # -------- WANDB LOG --------
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "lr": optimizer.param_groups[0]["lr"]
        })

        # -------- EARLY STOPPING --------
        if val_acc > best_val_acc:

            best_val_acc = val_acc
            patience_counter = 0

            torch.save(model.state_dict(), "best_model.pth")

            print("Model Improved. Saved.")

        else:

            patience_counter += 1

            print(f"Early stopping counter: {patience_counter}/{patience}")

            if patience_counter >= patience:

                print("Early stopping triggered")
                break

    print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")

# Training execution¶

In [ ]:
epochs= 25
patience= 3

In [ ]:
train_model(model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=device,
        epochs=EPOCHS,
        patience=patience
    )

# Model upload to kagglehub

In [ ]:
model_path = "/kaggle/working/Scratch_CNN.pth"

'''# have used DataParallel, saving the module's weights'''
torch.save(model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict(), model_path)

print("Model saved locally at:", model_path)

In [ ]:
KAGGLE_USERNAME = "uditmaurya1588"     # your Kaggle username
MODEL = "models"              # project/model name
FRAMEWORK = "pytorch"                  # framework
VARIATION = "ScratchCNN-v1"              # version or hyperparameter variant

handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"

In [ ]:
kagglehub.model_upload(
    handle,
    model_path,
    version_notes="Initial model version"
)

# Inference Code

In [ ]:
model = FinetunedCNN().to(device)


model_file = '/kaggle/input/models/pytorch/ScratchCNN-v1/1/Scratch_CNN.pth'
state_dict = torch.load(model_file, map_location=device)

# fix 'module.' prefix if trained with DataParallel
if any(k.startswith("module.") for k in state_dict.keys()):
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

model.load_state_dict(state_dict)

print("Model loaded and ready for predictions.")

In [ ]:
def audio_to_mel(path):

    audio, _ = librosa.load(path, sr=SR)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    # crop / pad to fixed width
    if mel.shape[1] > TARGET_WIDTH:
        mel = mel[:, :TARGET_WIDTH]
    else:
        mel = np.pad(mel, ((0,0),(0, TARGET_WIDTH - mel.shape[1])))

    mel = librosa.power_to_db(np.maximum(mel, 1e-10))

    # normalize
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    return mel.astype(np.float32)

In [ ]:
class MashupTestDataset(Dataset):

    def __init__(self, test_csv, mashup_dir):

        self.df = pd.read_csv(test_csv)
        self.mashup_dir = mashup_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sample_id = str(row["id"]).zfill(4)

        filename = row["filename"] if "filename" in row else sample_id + ".wav"

        path = os.path.join(self.mashup_dir, filename)

        mel = audio_to_mel(path)

        mel = torch.tensor(mel).unsqueeze(0)

        return mel, sample_id

In [ ]:
test_csv = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
mashup_dir = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"

test_dataset = MashupTestDataset(test_csv, mashup_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4   
)

In [ ]:
genres = [
    "blues","classical","country","disco","hiphop",
    "jazz","metal","pop","reggae","rock"
]

predictions = []

model.eval()

with torch.no_grad():

    for mel, ids in tqdm(test_loader, desc="Predicting"):

        mel = mel.to(device)

        outputs = model(mel)

        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        for sample_id, pred in zip(ids, preds):
            predictions.append((sample_id, genres[pred]))

In [ ]:
import pandas as pd

submission = pd.DataFrame(predictions, columns=["id", "genre"])

submission.to_csv("submission.csv", index=False)


In [ ]:
submission